# L15a: Autotrader Diagnostics, Risk Controls, and Validation
In this lecture, we evaluate an adaptive trading system as an engineered decision process: parameters may drift, performance must be compared with declared baselines, and deployment must be conditional on explicit validation gates.

> __Learning Objectives:__
>
> By the end of this lecture, you will be able to:
>
> * __Update a Single Index Model online:__ Use exponentially weighted least squares to revise SIM parameters while controlling the effective memory of the estimator.
> * __Diagnose strategy behavior:__ Compare frozen and adaptive configurations using paired paths, drawdown, failure rate, turnover, tail loss, and net present value.
> * __Apply deployment gates:__ Convert quantitative limits into a transparent pass/fail validation report and a pre-approved risk-control configuration.

Let's determine whether the Autotrader is ready to operate—and under what limits.
___


## Examples
The validation workflow will use two coordinated examples:

> [▶ Update the single index model and the preference weights](CHEME-5660-L15a-Example-Updating-SIM-Gamma-Fall-2026.ipynb). In this example, we update the single index model of every stock with each trading day of 2025 and 2026, starting from the 2014 to 2024 fit. We compute the preference weights from the updated estimates and pick the preference basket once a month and whenever the market turns. Then we compare the preference basket with the basket from the fixed 2014 to 2024 estimates.
>
> [▶ Build an Autotrader validation report](./CHEME-5660-L15a-Example-Validation-Gates-Fall-2026.ipynb). In this example, we construct paired validation evidence, evaluate explicit pass/fail gates, and produce a compact risk-control configuration.

The examples separate model adaptation from the governance decision to permit deployment.
___


## Notation
We use the course conventions established in the SIM and portfolio units:

> * $g_{m,t}$ and $g_{i,t}$ denote the market and asset continuously compounded growth rates at observation $t$.
> * $\alpha_i$, $\beta_i$, and $\sigma_{\varepsilon,i}$ denote the SIM intercept, market exposure, and residual standard deviation for asset $i$.
> * $W_{p,T}$ and $W_{b,T}$ denote terminal strategy and benchmark wealth on validation path $p$.

The half-life and the validation thresholds are model inputs and must be reported with the results. The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L15a) lists the symbols of the updating section with their meanings and units.
___


## Concept Review: From Adaptive Allocation to Adaptive Estimation
L13a allowed portfolio preferences and target weights to respond to a market-state signal, but the SIM parameters feeding that process remained fixed. A frozen calibration is reasonable only while the relationship between each asset and the market remains sufficiently stable.

Online estimation adds a second adaptation layer. It can improve responsiveness when exposures change, but it also introduces estimator noise, half-life sensitivity, and additional ways for the strategy to overreact. Therefore, the online and frozen configurations must be compared on common paths under the same costs and risk limits.

The engineering question is not simply whether the adaptive estimate moves. It is whether that movement improves the declared portfolio objectives without violating the operating envelope.
___


## Updating the Single Index Model
Previously, we fit the single index model of each stock once, on the 2014 to 2024 data, and held its estimates fixed. However, a stock's market sensitivity changes over time. By late 2026, most stocks in our universe moved less with the market than their 2014 to 2024 estimates say. We keep the estimates current by updating them with each new trading day, giving recent days the most weight:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L15a-EWLS-JPM.svg" width="740" style="max-width:100%; height:auto;" alt="The market sensitivity of JPM over 2025 and 2026. A dashed line marks the fixed estimate from the 2014 to 2024 fit, 1.11. The blue updated estimate, with one update per trading day, starts at 1.11, stays near 1.0 through 2025, and drops to about 0.7 in mid-2026. Late in 2026, JPM moves less with the market than the fixed estimate says."/>
    </center>
</div>

Previously, we estimated $\alpha_{i}$ and $\beta_{i}$ by least squares, where we minimized the sum of squared residuals with every training day counted the same. However, we could include __new information__ as it arrives by weighting the new days more than the old ones. Let's consider exponentially weighted least squares (EWLS):

> __Exponentially weighted least squares (EWLS):__ On day $t$, the updated baseline $\alpha_{i}(t)$ and market sensitivity $\beta_{i}(t)$ of stock $i$ are the solutions to the weighted least-squares problem:
>
> $$
> \left(\alpha_{i}(t),\beta_{i}(t)\right) = \arg\min_{\alpha_{i},\,\beta_{i}}\Biggl[\underbrace{\sum_{k=1}^{t}\phi^{\,t-k}\,\bigl(\overbrace{g_{i}(k)-\alpha_{i}-\beta_{i}\,g_{m}(k)}^{\text{residual }r_{i}(k)}\bigr)^{2}}_{\text{new days}} + \underbrace{\phi^{\,t}\,\frac{N_{0}}{2766}\sum_{k=-2765}^{0}r_{i}(k)^{2}}_{\text{prior}}\Biggr]
> $$
>
> where $g_{i}(k)$ and $g_{m}(k)$ are the daily growth rates of stock $i$ and `SPY` on day $k$ (1/yr). Day $0$ is December 31, 2024, the last day of the 2014 to 2024 fit, day $1$ is January 2, 2025, and day $469$ is November 13, 2026, the last day of our data.
>
> The problem has three parts:
>
> * __New days:__ The weight of day $k$ is the decay $\phi=2^{-1/63}\approx 0.989$ raised to the age of the day, $t-k$ trading days. For example, on day $3$, January 6, 2025, the first sum is $\phi^{2}\,r_{i}(1)^{2}+\phi\,r_{i}(2)^{2}+r_{i}(3)^{2}$. The weight of a day halves every 63 trading days, about three months.
> * __Prior:__ Each of the 2,766 days of 2014 to 2024 gets the weight $N_{0}/2766$, with $N_{0}=63$. On day $0$, the prior alone gives the 2014 to 2024 fit. After that, the factor $\phi^{\,t}$ shrinks the prior's day $0$ weight.
> * __No look-ahead:__ The estimates on day $t$ use the growth rates through day $t$ and nothing later. The trading method decides after the close of day $t$ and trades at the next open. However, this is still a backward-looking estimate.

Details of the implementation, which updates the estimates from a few running sums, are shown in [the EWLS recursion notebook](advanced/ewls/CHEME-5660-L15a-Derivation-EWLS-Update-Fall-2026.ipynb).

### Updating the General Factor Model
Previously, we fit the general factor model once, with every training day counted the same. The same exponential weights can update it with each new trading day:

* __Market, economic, and sector loadings:__ Steps 1 and 3 of the fitting algorithm become weighted least-squares fits. On day $t$, they use the weights of the EWLS problem, including the prior from the 2014 to 2024 fit.
* __Sector factors:__ Step 2 computes the sector factors of day $k$ from the residuals of that day alone. It needs no weights.
* __Factor averages and covariance:__ The averages $\boldsymbol{\mu}$ and the covariance matrix $\mathbf{\Sigma}_{Y}$ of the market, economic, and sector factors become exponentially weighted, with the same weights, before the Cholesky step.

Refitting the jump hidden Markov models is the slow part of an update. They can be refit less often than the loadings.

### Preference Weights with Updated Estimates
Previously, we computed the preference weight $\gamma_{i}(t)$ of each stock and used it to rank the stocks for inclusion in the portfolio. The preference weight depends on the market state $\xi_{t}$, which compares a 21-day and a 63-day EMA of the `SPY` price:

$$
\xi_{t} = -G\left(\frac{\bar{S}^{\text{short}}_{t}}{\bar{S}^{\text{long}}_{t}}-1\right)
$$

where the gain $G>0$ scales $\xi_{t}$ so that $|\xi_{t}|\leq 1$ on 99% of training days. When the market falls, recent prices sit below the quarter's average, and $\xi_{t}$ turns positive. The day $\xi_{t}$ changes sign is a market turn, the regime change that the trading method acts on.

The estimates and the preference weights update every trading day, but the basket of stocks we hold changes only on pick days:

<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L15a-Gamma-Picks.svg" width="860" style="max-width:100%; height:auto;" alt="One time axis with three rows. The top row is a path of the market state xi t, shaded blue where the market is rising and red where it is falling. The middle row is a line with a dot for every trading day, when alpha, beta, and gamma update. The bottom row shows the preference baskets held, numbered 1 to 7, each from one pick day to the next. Most baskets last 21 trading days. A brace over basket 2 reads: 15 trading days, cut short by the market turn. Where xi t crosses zero, an orange dashed line marks a market turn: the method picks a new basket, and the 21-day count restarts."/>
    </center>
</div>

The trading method of [Varner (2026)](https://arxiv.org/abs/2611.03022) uses the same preference weight to choose its stocks:

> __Preference model (updated estimates):__ For stock $i$ on day $t$, the preference weight is given by:
>
> $$
> \boxed{\begin{aligned}
> \gamma_{i}(t) &= \tanh\left(h\,\frac{\tilde{g}_{i,t}}{\max\left(|\beta_{i}(t)|,\,\beta_{\text{floor}}\right)^{\xi_{t}}}\right)\\
> &= \tanh\left(h\,\frac{\alpha_{i}(t)+\beta_{i}(t)\,\tilde{g}_{M,t}}{\max\left(|\beta_{i}(t)|,\,\beta_{\text{floor}}\right)^{\xi_{t}}}\right)
> \end{aligned}}
> $$
>
> where $\tilde{g}_{i,t}$ is the expected growth rate of stock $i$ in today's market (1/yr), the horizon $h=1$ year makes the argument of the $\tanh$ dimensionless, and the floor is $\beta_{\text{floor}}=0.1$. With a fixed $\beta_{i}\geq\beta_{\text{floor}}$, this is the preference weight we used before.
>
> The second line computes $\tilde{g}_{i,t}$ with the single index model, where the recent market growth rate $\tilde{g}_{M,t}$ (1/yr) is a 252-day EMA of the market growth rates. The single index model is one choice. A different factor model, or the stock's past growth rates alone, could give $\tilde{g}_{i,t}$ instead.
>
> The trading method makes three changes:
>
> * __Updated estimates:__ The estimates $\alpha_{i}(t)$ and $\beta_{i}(t)$ come from the daily EWLS update. The preference weight follows the changes in each stock's market sensitivity. AQR's [betting-against-beta portfolio](https://www.aqr.com/Insights/Research/Journal-Article/Betting-Against-Beta) also re-estimates the market sensitivity of each stock from a rolling window of past returns.
> * __A floor on the market sensitivity:__ Some stocks have an updated market sensitivity near or below zero. The denominator uses $|\beta_{i}(t)|$ with a floor, which keeps the denominator positive for every stock. The sign of the preference weight is then the sign of the expected growth rate $\tilde{g}_{i,t}$.
> * __Picks at market turns:__ The __preference basket__ holds the stocks with the highest preference weights in each sector. We choose how many stocks to keep from each sector. The method picks a new basket every 21 trading days and whenever the market turns, but other triggers are also possible.

### The Market Hedge
The trading method can also run with a __market hedge__, which limits the market sensitivity of the whole portfolio. The limit drops as the market falls:

<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L15a-Market-Hedge.svg" width="720" style="max-width:100%; height:auto;" alt="A line shows the limit on the market sensitivity of the portfolio, beta P of t, against the market state xi t. The limit falls from 2 at xi t equals minus 1 to 1 at xi t equals 0, the market sensitivity of SPY. It reaches 0 at xi t equals 1, the market sensitivity of betting against beta. Below zero, a red area marks where the method sells SPY short, and the line flattens at a floor of minus 0.5. Arrows from the line point down: beta P of t stays on or below the line. A gray band marks where the absolute value of xi t is at most 1, which holds on 99 percent of training days."/>
    </center>
</div>

> __Market hedge:__ On each day $t$ that the trading method sets its holdings, the hedge requires:
>
> $$
> \boxed{\beta_{P}(t) = \sum_{i}w_{i}(t)\,\beta_{i}(t) \leq \max\left(1-\xi_{t},\,-0.5\right)}
> $$
>
> where $\beta_{P}(t)$ is the market sensitivity of the portfolio, and $w_{i}(t)$ is the fraction of the portfolio's value in holding $i$, negative for a short. Cash has a market sensitivity of zero. Let's consider a few cases:
>
> * __Market rise:__ When the market rises ($\xi_{t}<0$), the limit on $\beta_{P}(t)$ is above one, so the hedge allows more market risk than `SPY`. Because the method never borrows to buy stocks, $\beta_{P}(t)$ rises above one only when the stocks it holds move more than the market on average.
> * __Flat or falling market:__ In a flat market ($\xi_{t}=0$), the limit on $\beta_{P}(t)$ is one, the market sensitivity of `SPY`. As the market falls, the limit drops to zero at $\xi_{t}=1$. To keep $\beta_{P}(t)$ at or below the limit, the method can move money into the basket's stocks with a low market sensitivity, hold cash, or buy up to five stocks whose market sensitivity is zero or negative.
> * __Steep fall:__ After a steep fall ($\xi_{t}>1$), the limit is below zero. The method then holds stocks and cash that are uncorrelated or anti-correlated with the market, and sells `SPY` short to push $\beta_{P}(t)$ below zero. Because `SPY` has a market sensitivity of one, a short worth 20% of the portfolio's value lowers $\beta_{P}(t)$ by $0.2$.
>
> The floor of $-0.5$ on the limit caps the `SPY` short at half of the portfolio's value.


Let's update the estimates of every stock in our universe and pick the preference basket:

> __Review on your own:__
>
> [▶ Update the single index model and the preference weights](CHEME-5660-L15a-Example-Updating-SIM-Gamma-Fall-2026.ipynb). In this example, we update the single index model of every stock with each trading day of 2025 and 2026, starting from the 2014 to 2024 fit. We pick the preference basket once a month and whenever the market turns, and compare it with the basket from the fixed estimates.

The example ends with the preference basket of the last pick day in our data.

__How does the trading method compare with betting against beta?__ Previously, we met [AQR Capital Management](https://www.aqr.com/Our-Firm/About-Us), where Andrea Frazzini and Lasse H. Pedersen proposed the [betting-against-beta portfolio](https://www.aqr.com/Insights/Research/Journal-Article/Betting-Against-Beta) in 2014. Their portfolio always buys stocks with a low market sensitivity and sells stocks with a high one, in amounts that make its market sensitivity zero. The trading method makes the same bet only when the market falls:

* __Signal__: When the market falls ($\xi_{t}>0$), the denominator is above one for a stock with $|\beta_{i}(t)|>1$ and below one for a stock with $|\beta_{i}(t)|<1$. Among stocks with a positive expected growth rate, the preference weight then favors stocks that move less with the market. When the market rises ($\xi_{t}<0$), the tilt reverses.
* __Long-only basket__: The preference basket keeps the stocks with the highest preference weights in each sector. A stock with a low preference weight is left out, not sold short.
* __Market hedge__: With the hedge on, the limit on $\beta_{P}(t)$ reaches zero, the market sensitivity of betting against beta, at $\xi_{t}=1$. Betting against beta reaches zero by selling stocks with a high market sensitivity. The trading method sells only `SPY` short, and only after a steeper fall takes the limit below zero.

___

## Validation and Compliance

Before the engine goes to production in the production-operations lecture, it must pass a formal validation. The validation report applies five pass/fail gates to the Monte Carlo backtest results:

| Criterion | Threshold | Rationale |
|-----------|-----------|-----------|
| **Median Sharpe** $\geq$ **0.3** | Risk-adjusted growth must exceed a low bar | Below 0.3, the strategy is not compensating for the risk it takes |
| **Median Max Drawdown** $\leq$ **25%** | Worst-case capital loss must be bounded | Larger drawdowns trigger investor redemptions and regulatory scrutiny |
| **Failure Rate** $\leq$ **10%** | At most 10% of paths end below initial capital | Systematic losses indicate a flawed strategy |
| **Wealth versus Frozen Baseline** $\geq$ **1.0** | Median terminal wealth must beat the frozen-parameter engine | An online learner that does not beat its frozen ancestor is not earning its complexity |
| **Median NPV** $\geq$ **0** | Median path must beat the risk-free baseline in present-value terms | A strategy whose median NPV is negative is dominated by a T-bill |

A strategy that passes all five gates across model-generated paths has demonstrated robustness to regime shifts, fat tails, and volatility clustering. It is not guaranteed to work in production, but it has cleared the minimum bar for deployment.

The validation report also produces a set of **compliance parameters** that become the pre-approved operating limits for the production-operations lecture's production system:

* **Concentration cap:** maximum portfolio weight in any single asset (e.g., 40%)
* **Drawdown gate:** maximum drawdown before the engine de-risks to cash (e.g., 15%)
* **Turnover limit:** maximum daily trade volume as a fraction of portfolio value (e.g., 50%)
* **Position size limit:** maximum dollar exposure per individual trade (e.g., $5,000)

In the production-operations lecture, trades within these limits be eligible for automated execution; trades that exceed them are queued for human review. The compliance configuration is the bridge between _validated in simulation_ and _approved for production._

> __Example__
>
> [▶ Let's build the validation report and export compliance config](./CHEME-5660-L15a-Example-Validation-Gates-Fall-2026.ipynb). In this example, we evaluate the EWLS engine against the frozen Cobb-Douglas baseline across the five gates, produce pass/fail verdicts, and export the compliance parameters for the production-operations lecture.

___

## Optional Advanced Material
> [▶ Derive the daily update of the single index model](advanced/ewls/CHEME-5660-L15a-Derivation-EWLS-Update-Fall-2026.ipynb). In this notebook, we derive the running sums that the exponentially weighted least squares update carries from one day to the next. We compute the updated estimates and the residual standard deviation from these sums, and show how the 2014 to 2024 fit sets the starting values of the sums.

___

## Summary
This lecture treated model adaptation and deployment authorization as separate engineering decisions.

> __Key Takeaways:__
>
> * __EWLS makes estimator memory explicit:__ The half-life controls how rapidly old observations lose influence.
> * __Adaptive and frozen models require paired evaluation:__ Common scenarios isolate policy differences from path differences.
> * __Deployment gates must be declared and auditable:__ Reward, drawdown, failure, benchmark, NPV, turnover, and operational limits each test a different failure mode.

Passing the classroom gates authorizes the captured production exercise only; it does not authorize unattended live trading.
___


## Disclaimer and Risks
This material is for educational purposes only and does not constitute investment advice. Passing a simplified validation gate does not establish that a model is suitable for live deployment.
